<h1> 1. Problem Statement

<h1>2. Import Library

In [ ]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

df = pd.read_csv('../data/customer_reviews_sentiment.csv - customer_reviews_sentiment.csv')
df.head()

<h2> 3. Train and Split (Model Classic)

In [ ]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

<h2> 4. Model Classic (Scikit-learn)

<h3> 4.1 Preprocessing & Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

<h3> 4.2 Training Model

In [ ]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

<h3 >4.3 Prediksi pada Test Set

In [ ]:
y_pred_classic = clf.predict(X_test_vec)

<h2> 5. LLM MODEL

<h3> 5.1 Setup API

In [ ]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

<h3> 5.2 Merancang Prompt

In [ ]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
def classify_sentiment_llm(review_text):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model='gemini-3.8-flash',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text.strip().lower()

<h3> 5.3 Menjalankan Prediksi pada Test Set

In [ ]:
y_pred_llm = []
for review in X_test:
    pred = classify_sentiment_llm(review)
    y_pred_llm.append(pred)

# TODO: normalisasi y_pred_llm agar konsisten dengan label 'positif'/'negatif'

<h2>6. Evaluasi dan Perbandingan

<h3 >6.1 Evaluasi Model Klasik

In [ ]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

<h3>6.2 Evaluasi LLM API

In [ ]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

<h1> 7. Analisis Trade-off dan Limitation


<h1>8. Rekomendasi Technical Approach